In [ ]:
#nieuwe area code

In [ ]:
import os
import pandas as pd
import imageio.v2 as imageio
import re
import numpy as np
from pathlib import Path
from tqdm import tqdm

# --- CONFIGURATION ---
image_root = Path(r"D:\Thesis\Screen crispri quality checked")

projects = [
    {"prefix": "PLATE1", "loc": Path(r"D:\Thesis\final\project_P1_neighbours_deduplication\inputs\locations")},
    {"prefix": "PLATE2", "loc": Path(r"D:\Thesis\final\project_P2_neighbours_deduplication\inputs\locations")},
    {"prefix": "PLATE3", "loc": Path(r"D:\Thesis\final\project_P3_neighbours_deduplication\inputs\locations")},
    {"prefix": "PLATE4", "loc": Path(r"D:\Thesis\final\project_P4_neighbours_deduplication\inputs\locations")},
    {"prefix": "PLATE5", "loc": Path(r"D:\Thesis\final\project_P5_neighbours_deduplication\inputs\locations")},
]

output_dir = Path(r"D:\Thesis\final\area")
os.makedirs(output_dir, exist_ok=True)

# Output paths
individual_cells_file = output_dir / "all_individual_cells_areas.csv"
site_medians_file = output_dir / "master_median_areas_per_site.csv"

mask_pattern = re.compile(r"Sample_(?P<well>[A-Z][0-9]+)_XY(?P<site>[0-9]+)_C1_cp_masks\.tif")

all_individual_data = []
all_site_medians = []

# --- PROCESSING ---
for proj in projects:
    loc_base = proj["loc"]
    prefix = proj["prefix"]
    
    if not loc_base.exists():
        print(f"Skipping {prefix}: Folder not found.")
        continue

    plate_folders = [d for d in os.listdir(loc_base) if d.startswith(prefix)]
    
    for plate_name in plate_folders:
        plate_loc_dir = loc_base / plate_name
        plate_img_dir = image_root / plate_name
        
        print(f"Processing {plate_name}...")
        csv_files = [f for f in os.listdir(plate_loc_dir) if f.endswith("-Nuclei.csv")]

        for csv_file in tqdm(csv_files, desc=f"  {plate_name}"):
            parts = csv_file.replace("-Nuclei.csv", "").split("-")
            well_id, site_id = parts[0], parts[1]
            
            try:
                df_coords = pd.read_csv(plate_loc_dir / csv_file)
                if df_coords.empty: continue
            except:
                continue

            well_folder = f"Sample_{well_id}"
            mask_folder = plate_img_dir / well_folder / "masks"
            
            mask_path = None
            if mask_folder.exists():
                for f in os.listdir(mask_folder):
                    if f"XY{site_id}_C1" in f and f.endswith(".tif"):
                        mask_path = mask_folder / f
                        break
            
            if not mask_path:
                continue

            # Load mask ONCE
            mask_data = imageio.imread(mask_path)
            labels, counts = np.unique(mask_data, return_counts=True)
            area_lookup = dict(zip(labels, counts))

            current_site_areas = []
            
            # Match coords to areas
            for _, row in df_coords.iterrows():
                x, y = int(round(row["Nuclei_Location_Center_X"])), int(round(row["Nuclei_Location_Center_Y"]))
                
                try:
                    label_id = mask_data[y, x]
                    if label_id > 0:
                        area = area_lookup.get(label_id, 0)
                        current_site_areas.append(area)
                        
                        # Store individual cell data
                        all_individual_data.append({
                            "Plate": plate_name,
                            "Well": well_id,
                            "Site": site_id,
                            "Area": area
                        })
                except IndexError:
                    continue

            # Store site median data
            if current_site_areas:
                all_site_medians.append({
                    "Plate": plate_name,
                    "Well": well_id,
                    "Site": site_id,
                    "Median_Area": np.median(current_site_areas),
                    "Cell_Count": len(current_site_areas)
                })

# --- SAVE EVERYTHING ---
print("\nSaving files...")

if all_individual_data:
    pd.DataFrame(all_individual_data).to_csv(individual_cells_file, index=False)
    print(f"  > Individual cells saved: {individual_cells_file}")

if all_site_medians:
    pd.DataFrame(all_site_medians).to_csv(site_medians_file, index=False)
    print(f"  > Site medians saved: {site_medians_file}")

print("\nDone!")

In [ ]:
import pandas as pd

# 1. Load your data
# Adding 'r' before the path tells Python to treat backslashes as literal text
input_path = r'D:\Thesis\final\area\master_median_areas_per_site.csv'
df = pd.read_csv(input_path)

# 2. Calculate the median area per well
# This groups all 'Sites' together for each unique Well
well_stats = df.groupby(['Plate', 'Well'])['Median_Area'].median().reset_index()

# 3. Save the results to a new CSV file
output_path = r'D:\Thesis\final\area\well_median_areas_output.csv'
well_stats.to_csv(output_path, index=False)

print(f"Success! Processed {len(well_stats)} wells.")
print(f"Results saved to: {output_path}")

In [ ]:
#well level median based on individual cells isntead of sites

In [ ]:
import pandas as pd

# 1. Load the individual cell data
# This file contains the 'Area' for every single cell matched in your previous script
input_path = r'D:\Thesis\final\area\all_individual_cells_areas.csv'
df_individual = pd.read_csv(input_path)

# 2. Calculate the median area per well
# We group by Plate and Well to ensure wells with the same name on different plates are kept separate
well_medians = df_individual.groupby(['Plate', 'Well'])['Area'].median().reset_index()

# 3. Optional: Add cell count per well
# This is often useful for quality control
well_counts = df_individual.groupby(['Plate', 'Well'])['Area'].count().reset_index()
well_counts.columns = ['Plate', 'Well', 'Cell_Count']

# Merge medians and counts
final_stats = pd.merge(well_medians, well_counts, on=['Plate', 'Well'])

# Rename the column for clarity
final_stats = final_stats.rename(columns={'Area': 'Well_Median_Area'})

# 4. Save the results
output_path = r'D:\Thesis\final\area\well_median_areas_from_cells.csv'
final_stats.to_csv(output_path, index=False)

print(f"Success! Processed {len(final_stats)} wells.")
print(f"Results saved to: {output_path}")

In [ ]:
#treatment column erbij

In [ ]:
import pandas as pd
import os

# --- CONFIGURATION ---
COORD_DIR = r'E:\Thesis3april\plots\All_Plates_NofeatureSelection\coordinates'
INDIVIDUAL_CELLS_PATH = r'D:\Thesis\final\area\all_individual_cells_areas.csv'
WELL_MEDIAN_PATH = r'D:\Thesis\final\area\well_median_areas_from_cells.csv'

# 1. Load the coordinate data to build the Treatment map
# We use the first file in the directory to extract the Well_ID -> Treatment relationship
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]
if not coord_files:
    print("No coordinate files found!")
else:
    df_coords = pd.read_csv(os.path.join(COORD_DIR, coord_files[0]))

    # Create the mapping dictionary
    # Since Well_ID is already "PLATE1_T1_B6", we can map it directly
    treatment_map = df_coords.set_index('Well_ID')['Treatment'].to_dict()

    # --- 2. UPDATE INDIVIDUAL CELLS FILE ---
    print("Updating individual cell areas...")
    df_indiv = pd.read_csv(INDIVIDUAL_CELLS_PATH)
    
    # Create the same ID format: "PLATE1_T1" + "_" + "B6"
    df_indiv['Well_ID_Match'] = df_indiv['Plate'] + "_" + df_indiv['Well']
    df_indiv['Treatment'] = df_indiv['Well_ID_Match'].map(treatment_map)
    
    # Save and cleanup
    df_indiv.drop(columns=['Well_ID_Match']).to_csv(INDIVIDUAL_CELLS_PATH, index=False)

    # --- 3. UPDATE WELL MEDIAN FILE ---
    print("Updating well median areas...")
    df_well = pd.read_csv(WELL_MEDIAN_PATH)
    
    df_well['Well_ID_Match'] = df_well['Plate'] + "_" + df_well['Well']
    df_well['Treatment'] = df_well['Well_ID_Match'].map(treatment_map)
    
    # Save and cleanup
    df_well.drop(columns=['Well_ID_Match']).to_csv(WELL_MEDIAN_PATH, index=False)

    print("Success! Treatment column added to both files using exact Plate_Well matching.")

In [ ]:
#oude area code

In [ ]:
import os
import pandas as pd
import imageio.v2 as imageio
import re
import numpy as np
from skimage import measure
from scipy.spatial import KDTree 

# --- CONFIGURATION ---
root_dir = "/media/arnout/Elements1/Thesis/Screen crispri quality checked"
loc_base = "/media/arnout/Elements1/Thesis/project_neighboursT20/inputs/locations"
# Where to save your personal area report
area_report_path = "/media/arnout/Elements1/Thesis/project_neighboursT20/cell_area_report.csv"

os.makedirs(os.path.dirname(area_report_path), exist_ok=True)

MIN_AREA_THRESHOLD = 700 
BOX_SIZE = 224
SQUARE_RADIUS = BOX_SIZE / 2  # 112 pixels
MAX_NEIGHBORS = 2  # Maximum neighbors allowed within the square box

mask_pattern = re.compile(r"Sample_(?P<well>[A-Z][0-9]+)_XY(?P<site>[0-9]+)_C1_cp_masks\.tif+")

found_count = 0
area_stats = [] # List to collect your personal data

# Plates to process
for plate in ["PLATE2_T0", "PLATE2_T1", "PLATE2_T2"]:
    if not plate.startswith("PLATE"): continue
    
    plate_loc_dir = os.path.join(loc_base, plate)
    os.makedirs(plate_loc_dir, exist_ok=True)
    
    plate_path = os.path.join(root_dir, plate)
    if not os.path.exists(plate_path):
        print(f"Warning: Plate path {plate_path} does not exist. Skipping.")
        continue
    
    print(f"Processing {plate}...")

    for well_folder in os.listdir(plate_path):
        if not well_folder.startswith("Sample_") or "bad_Q" in well_folder: continue
        mask_folder = os.path.join(plate_path, well_folder, "masks")
        
        if os.path.exists(mask_folder):
            for mask_name in os.listdir(mask_folder):
                match = mask_pattern.search(mask_name)
                if not match: continue
                
                well = match.group('well')
                site = match.group('site')
                out_filename = f"{well}-{site}-Nuclei.csv"
                
                mask_data = imageio.imread(os.path.join(mask_folder, mask_name))
                props = measure.regionprops(mask_data)
                
                # Filter by area threshold and extract properties
                # We keep the objects themselves here so we can access .area later
                valid_props = [p for p in props if p.area >= MIN_AREA_THRESHOLD]
                
                if not valid_props: continue

                # Extract coordinates for KDTree
                all_coords = np.array([[p.centroid[1], p.centroid[0]] for p in valid_props])

                # --- SPATIAL NEIGHBOR FILTERING ---
                tree = KDTree(all_coords)
                final_locations = []

                for i, prop in enumerate(valid_props):
                    # Square search using p=inf
                    neighbor_indices = tree.query_ball_point(all_coords[i], r=SQUARE_RADIUS, p=float('inf'))
                    neighbor_count = len(neighbor_indices) - 1
                    
                    if neighbor_count <= MAX_NEIGHBORS:
                        # 1. Add to DeepProfiler CSV list
                        final_locations.append({
                            "Nuclei_Location_Center_X": all_coords[i][0], 
                            "Nuclei_Location_Center_Y": all_coords[i][1]
                        })
                        
                        # 2. Add to your personal Area Report list
                        area_stats.append({
                            "Plate": plate,
                            "Well": well,
                            "Site": site,
                            "Area": prop.area
                        })

                if final_locations:
                    save_path = os.path.join(plate_loc_dir, out_filename)
                    pd.DataFrame(final_locations).to_csv(save_path, index=False)
                    found_count += 1
                    print(f"  > Created {out_filename} in {plate}: {len(final_locations)} cells kept")

# --- SAVE PERSONAL AREA REPORT ---
if area_stats:
    df_area = pd.DataFrame(area_stats)
    df_area.to_csv(area_report_path, index=False)
    print(f"\nPersonal area report saved to: {area_report_path}")

print(f"Success! Total location files generated: {found_count}")